The research question becomes:

Can explicit trajectory-state and action-relation features improve failure-family classification beyond semantic text embeddings?

In [1]:
%load_ext autoreload
%autoreload 2

import ast
import json
import re
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from scripts.taxonomy_dataset import (
    taxonomy_df,
    X_train,
    X_test,
    y_train,
    y_test,
    groups_train,
    groups_test,
)

print(taxonomy_df.shape)
print(taxonomy_df["failure_family"].value_counts())

DATASET
Total samples:       6,881
Total trajectories:  750
TRAIN / TEST SPLIT
Train samples:       5,491
Test samples:        1,390
Train trajectories:  600
Test trajectories:   150
TRAIN LABEL DISTRIBUTION
       count  percentage
label                   
-1      1457       26.53
 0       231        4.21
 1      3803       69.26
TEST LABEL DISTRIBUTION
       count  percentage
label                   
-1       402       28.92
 0        56        4.03
 1       932       67.05
LEAKAGE CHECK
Overlapping trajectories: 0
X / y / groups alignment: OK
Trajectory split:          OK
ANNOTATED FAILURES
Total: 1,859
dataset
A     179
B    1110
C     570
Name: count, dtype: int64

RULE TAXONOMY
failure_type
unknown                                640
repeated_action                        327
constraint_or_policy_violation         227
irrelevant_action                      111
missing_required_argument               62
unresolved_prior_error                  61
missing_required_action            

/Users/user/proj/agent-reliability-ml/notebooks/scripts/taxonomy_dataset.py:673: RuntimeWarning: Rule-stage unknown count: notebook checkpoint was 608, runtime reconstruction produced 640. Continuing through embedding/similarity resolution; the final training dataset is asserted exactly.
  _notebook_checkpoint(


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

/Users/user/proj/agent-reliability-ml/notebooks/scripts/taxonomy_dataset.py:885: RuntimeWarning: Prototype category count: notebook checkpoint was 23, runtime reconstruction produced 20. Continuing through embedding/similarity resolution; the final training dataset is asserted exactly.
  _notebook_checkpoint(
/Users/user/proj/agent-reliability-ml/notebooks/scripts/taxonomy_dataset.py:937: RuntimeWarning: Reference-bank size: notebook checkpoint was 771, runtime reconstruction produced 758. Continuing through embedding/similarity resolution; the final training dataset is asserted exactly.
  _notebook_checkpoint(


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/user/proj/agent-reliability-ml/.venv/lib/python3.12/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
/Users/user/proj/agent-reliability-ml/notebooks/scripts/taxonomy_dataset.py:1262: RuntimeWarning: Remaining KMeans cl

\nNOTEBOOK REPRODUCTION CHECK
Runtime five-family samples: 1,776
Notebook reference samples: 1,779
\nRuntime family counts:
failure_family
workflow_error           798
constraint_error         387
tool_use_error           275
grounding_state_error    274
reasoning_value_error     42
Name: count, dtype: int64
\nNotebook reference family counts:
workflow_error           789
constraint_error         397
tool_use_error           274
grounding_state_error    264
reasoning_value_error     55
dtype: int64

TAXONOMY DATASET
Total samples:       1,776
Total trajectories:  419

TRAIN / TEST SPLIT
Train samples:       1,489
Test samples:        287
Train trajectories:  335
Test trajectories:   84

TRAIN LABEL DISTRIBUTION
              count  percentage
family_label                   
0               660       44.33
1               317       21.29
2               237       15.92
3               244       16.39
4                31        2.08

TEST LABEL DISTRIBUTION
              count  percentag

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Train embeddings: (1489, 384)
Train labels:     (1489,)
Test embeddings:  (287, 384)
Test labels:      (287,)
(1776, 30)
failure_family
workflow_error           798
constraint_error         387
tool_use_error           275
grounding_state_error    274
reasoning_value_error     42
Name: count, dtype: int64


In [2]:
print(X_train.index[:10])
print(X_test.index[:10])

print("Train:", len(X_train))
print("Test:", len(X_test))
print("Group overlap:", len(set(groups_train) & set(groups_test)))

RangeIndex(start=0, stop=10, step=1)
RangeIndex(start=0, stop=10, step=1)
Train: 1489
Test: 287
Group overlap: 0


In [3]:
def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x)


def basic_structural_features(df):
    out = pd.DataFrame(index=df.index)

    # Position / role
    out["message_index"] = df["message_index"].fillna(0)
    out["is_tool_call"] = df["is_tool_call"].fillna(0).astype(int)

    out["current_role"] = (
        df["current_role"]
        .fillna("UNKNOWN")
        .astype(str)
    )

    # Length information
    out["current_char_length"] = (
        df["current_text"]
        .map(lambda x: len(safe_text(x)))
    )

    out["current_word_count"] = (
        df["current_text"]
        .map(lambda x: len(safe_text(x).split()))
    )

    out["context_char_length"] = (
        df["context_text"]
        .map(lambda x: len(safe_text(x)))
    )

    out["context_word_count"] = (
        df["context_text"]
        .map(lambda x: len(safe_text(x).split()))
    )

    # Historical channel sizes
    history_cols = [
        "previous_messages",
        "previous_tool_calls",
        "previous_tool_results",
        "previous_user_messages",
        "previous_assistant_messages",
    ]

    for col in history_cols:
        out[f"{col}_chars"] = (
            df[col]
            .map(lambda x: len(safe_text(x)))
        )

    return out

In [4]:
structured_df = basic_structural_features(taxonomy_df)

structured_df.head()

,message_index,is_tool_call,current_role,current_char_length,current_word_count,context_char_length,context_word_count,previous_messages_chars,previous_tool_calls_chars,previous_tool_results_chars,previous_user_messages_chars,previous_assistant_messages_chars
0,8,0,ASSISTANT,855,137,2499,341,1,1,1,1,1
1,6,0,ASSISTANT,628,101,2334,345,1,1,1,1,1
2,10,0,ASSISTANT,39,4,2423,350,1,1,1,1,1
3,2,0,ASSISTANT,600,92,0,0,1,1,1,1,1
4,2,0,ASSISTANT,933,150,0,0,1,1,1,1,1


In [5]:
structured_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
message_index,1776.0,NaN,NaN,NaN,34.183559,37.773768,2.0,11.0,21.0,39.0,203.0
is_tool_call,1776.0,NaN,NaN,NaN,0.552928,0.497331,0.0,0.0,1.0,1.0,1.0
current_role,1776,2,TOOL_CALL,982,NaN,NaN,NaN,NaN,NaN,NaN,NaN
current_char_length,1776.0,NaN,NaN,NaN,380.906532,475.129781,11.0,52.0,188.5,547.25,4358.0
current_word_count,1776.0,NaN,NaN,NaN,56.675676,79.034994,1.0,3.0,23.0,82.0,657.0
context_char_length,1776.0,NaN,NaN,NaN,1036.108108,1810.166995,0.0,146.75,352.0,1000.5,18222.0
context_word_count,1776.0,NaN,NaN,NaN,133.778153,260.922401,0.0,11.0,29.0,121.0,2619.0
previous_messages_chars,1776.0,NaN,NaN,NaN,1.724099,0.567082,1.0,1.0,2.0,2.0,3.0
previous_tool_calls_chars,1776.0,NaN,NaN,NaN,1.334459,0.471934,1.0,1.0,1.0,2.0,2.0
previous_tool_results_chars,1776.0,NaN,NaN,NaN,1.334459,0.471934,1.0,1.0,1.0,2.0,2.0


In [7]:
TOOL_CALL_PATTERN = re.compile(
    r"\[TOOL_CALL\]\s*"
    r"([A-Za-z_][A-Za-z0-9_]*)\s*\(",
    re.MULTILINE,
)


def extract_tool_names(text):
    text = safe_text(text)

    return TOOL_CALL_PATTERN.findall(text)


def current_tool_name(text):
    names = extract_tool_names(text)

    if not names:
        return "NO_TOOL"

    return names[-1]

In [8]:
examples = taxonomy_df.loc[
    taxonomy_df["is_tool_call"] == 1,
    "current_text"
].head(10)

for text in examples:
    print(current_tool_name(text))
    print(text[:200])
    print("-" * 80)

search
[TOOL_CALL]

search({"query_list": ["Cressida Bonas Douglas Smith Lucien Laviscount 2017 film cast"]})
--------------------------------------------------------------------------------
search
[TOOL_CALL]

search({"query_list": ["Cressida Bonas 2017 film with Douglas Smith and Lucien Laviscount"]})
--------------------------------------------------------------------------------
search
[TOOL_CALL]

search({"query_list": ["East Lothian Scotland coastal area south side Dirleton Castle"]})
--------------------------------------------------------------------------------
NO_TOOL
[TOOL_CALL]
The search results show Dirleton lies between North Berwick (east), Gullane (west), Fenton Barns (south) and the Yellowcraigs nature reserve, Archerfield Estate, and the Firth of Forth (no
--------------------------------------------------------------------------------
NO_TOOL
[TOOL_CALL]
</think>Based on my searches, I need to determine what coastal area Dirleton Castle borders on the south side. Th

In [9]:
structured_df["current_tool"] = (
    taxonomy_df["current_text"]
    .map(current_tool_name)
)

In [11]:
structured_df["current_tool"].value_counts().head(30)

current_tool
NO_TOOL                                  918
get_details_by_id                        197
search                                    56
book_reservation                          28
book_flight                               28
echo                                      27
get_customer_by_phone                     23
cd                                        22
transfer_to_human_agents                  22
get_data_usage                            21
check_network_status                      21
check_status_bar                          21
check_sim_status                          19
check_apn_settings                        15
check_data_restriction_status             14
check_network_mode_preference             14
ls                                        13
update_reservation_flights                13
check_vpn_status                          13
search_direct_flight                      12
search_onestop_flight                     12
check_wifi_calling_status                 

In [12]:
def previous_tool_name(text):
    names = extract_tool_names(text)

    if not names:
        return "NO_TOOL"

    return names[-1]


structured_df["previous_tool"] = (
    taxonomy_df["context_text"]
    .map(previous_tool_name)
)

In [13]:
structured_df["same_tool_as_previous"] = (
    (
        structured_df["current_tool"]
        == structured_df["previous_tool"]
    )
    &
    (
        structured_df["current_tool"]
        != "NO_TOOL"
    )
).astype(int)

In [14]:
structured_df[
    [
        "current_tool",
        "previous_tool",
        "same_tool_as_previous",
    ]
].head(30)

,current_tool,previous_tool,same_tool_as_previous
0,NO_TOOL,search,0
1,NO_TOOL,search,0
2,NO_TOOL,NO_TOOL,0
3,NO_TOOL,NO_TOOL,0
4,NO_TOOL,NO_TOOL,0
5,NO_TOOL,NO_TOOL,0
6,NO_TOOL,search,0
7,NO_TOOL,NO_TOOL,0
8,search,search,1
9,search,search,1


In [15]:
structured_df["num_previous_tool_calls"] = (
    taxonomy_df["context_text"]
    .map(
        lambda x: len(
            extract_tool_names(x)
        )
    )
)

In [16]:
def count_current_tool_in_context(row):
    current_tool = current_tool_name(
        row["current_text"]
    )

    if current_tool == "NO_TOOL":
        return 0

    previous_tools = extract_tool_names(
        row["context_text"]
    )

    return previous_tools.count(current_tool)


structured_df["current_tool_previous_count"] = (
    taxonomy_df.apply(
        count_current_tool_in_context,
        axis=1,
    )
)

In [17]:
def normalize_action(text):
    text = safe_text(text)

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().lower()

In [18]:
def current_action_in_context(row):
    current = normalize_action(
        row["current_text"]
    )

    context = normalize_action(
        row["context_text"]
    )

    if not current:
        return 0

    return int(current in context)


structured_df["current_action_seen_before"] = (
    taxonomy_df.apply(
        current_action_in_context,
        axis=1,
    )
)

In [19]:
ERROR_TERMS = [
    "error",
    "failed",
    "failure",
    "invalid",
    "not found",
    "unavailable",
    "denied",
    "cannot",
    "can't",
    "missing",
    "required",
]


def contains_error_signal(text):
    text = safe_text(text).lower()

    return int(
        any(
            term in text
            for term in ERROR_TERMS
        )
    )

In [20]:
structured_df["previous_results_have_error"] = (
    taxonomy_df["previous_tool_results"]
    .map(contains_error_signal)
)

In [21]:
structured_df["context_has_error_signal"] = (
    taxonomy_df["context_text"]
    .map(contains_error_signal)
)

In [22]:
def has_nonempty_text(x):
    return int(
        bool(
            safe_text(x).strip()
        )
    )


structured_df["has_previous_tool_result"] = (
    taxonomy_df["previous_tool_results"]
    .map(has_nonempty_text)
)

structured_df["has_previous_tool_call"] = (
    taxonomy_df["previous_tool_calls"]
    .map(has_nonempty_text)
)

structured_df["has_previous_user_message"] = (
    taxonomy_df["previous_user_messages"]
    .map(has_nonempty_text)
)

In [25]:
feature_columns = [
    # categorical
    "current_role",
    "current_tool",
    "previous_tool",

    # numerical
    "message_index",
    "is_tool_call",

    "current_char_length",
    "current_word_count",
    "context_char_length",
    "context_word_count",

    "previous_messages_chars",
    "previous_tool_calls_chars",
    "previous_tool_results_chars",
    "previous_user_messages_chars",
    "previous_assistant_messages_chars",

    "num_previous_tool_calls",
    "current_tool_previous_count",

    # relational / binary
    "same_tool_as_previous",
    "current_action_seen_before",
    "previous_results_have_error",
    "context_has_error_signal",
    "has_previous_tool_result",
    "has_previous_tool_call",
    "has_previous_user_message",
]

structured_df = structured_df[
    feature_columns
].copy()

print(structured_df.shape)

structured_df.head(20)

(1776, 23)


,current_role,current_tool,previous_tool,message_index,is_tool_call,current_char_length,current_word_count,context_char_length,context_word_count,previous_messages_chars,...,previous_assistant_messages_chars,num_previous_tool_calls,current_tool_previous_count,same_tool_as_previous,current_action_seen_before,previous_results_have_error,context_has_error_signal,has_previous_tool_result,has_previous_tool_call,has_previous_user_message
0,ASSISTANT,NO_TOOL,search,8,0,855,137,2499,341,1,...,1,1,0,0,0,0,0,1,1,1
1,ASSISTANT,NO_TOOL,search,6,0,628,101,2334,345,1,...,1,1,0,0,0,0,0,1,1,1
2,ASSISTANT,NO_TOOL,NO_TOOL,10,0,39,4,2423,350,1,...,1,0,0,0,0,0,0,1,1,1
3,ASSISTANT,NO_TOOL,NO_TOOL,2,0,600,92,0,0,1,...,1,0,0,0,0,0,0,1,1,1
4,ASSISTANT,NO_TOOL,NO_TOOL,2,0,933,150,0,0,1,...,1,0,0,0,0,0,0,1,1,1
5,ASSISTANT,NO_TOOL,NO_TOOL,2,0,320,49,0,0,1,...,1,0,0,0,0,0,0,1,1,1
6,ASSISTANT,NO_TOOL,search,6,0,619,98,4700,674,1,...,1,1,0,0,0,0,1,1,1,1
7,ASSISTANT,NO_TOOL,NO_TOOL,4,0,1929,322,9302,1427,1,...,1,0,0,0,0,0,0,1,1,1
8,TOOL_CALL,search,search,8,1,102,11,2257,334,1,...,1,1,1,1,0,0,0,1,1,1
9,TOOL_CALL,search,search,10,1,106,12,2258,337,1,...,1,1,1,1,0,0,0,1,1,1


In [26]:
structured_train = structured_df.loc[
    X_train.index
].copy()

structured_test = structured_df.loc[
    X_test.index
].copy()

print(structured_train.shape)
print(structured_test.shape)

assert len(structured_train) == len(y_train)
assert len(structured_test) == len(y_test)

(1489, 23)
(287, 23)


In [28]:
categorical_features = [
    "current_role",
    "current_tool",
    "previous_tool",
]

numeric_features = [
    col
    for col in feature_columns
    if col not in categorical_features
]

In [29]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

In [30]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        ),
    ]
)

structured_classifier = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight=None,
            ),
        ),
    ]
)

In [33]:
structured_classifier.fit(
    structured_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](5,)","[0,1,2,3,4]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['current_role','current_tool','previous_tool',..., 'has_previous_tool_result','has_previous_tool_call', 'has_previous_user_message']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns

In [34]:
structured_pred = (
    structured_classifier.predict(
        structured_test
    )
)

In [35]:
family_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

print(
    classification_report(
        y_test,
        structured_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

structured_accuracy = accuracy_score(
    y_test,
    structured_pred,
)

structured_balanced_accuracy = (
    balanced_accuracy_score(
        y_test,
        structured_pred,
    )
)

structured_macro_f1 = f1_score(
    y_test,
    structured_pred,
    average="macro",
)

structured_weighted_f1 = f1_score(
    y_test,
    structured_pred,
    average="weighted",
)

print(
    "Accuracy:",
    structured_accuracy,
)

print(
    "Balanced accuracy:",
    structured_balanced_accuracy,
)

print(
    "Macro F1:",
    structured_macro_f1,
)

print(
    "Weighted F1:",
    structured_weighted_f1,
)

                       precision    recall  f1-score   support

       workflow_error     0.4579    0.6304    0.5305       138
     constraint_error     0.1486    0.1571    0.1528        70
       tool_use_error     0.0000    0.0000    0.0000        38
grounding_state_error     0.0000    0.0000    0.0000        30
reasoning_value_error     0.0000    0.0000    0.0000        11

             accuracy                         0.3415       287
            macro avg     0.1213    0.1575    0.1367       287
         weighted avg     0.2564    0.3415    0.2923       287

Accuracy: 0.34146341463414637
Balanced accuracy: 0.15751552795031057
Macro F1: 0.13665311653116533
Weighted F1: 0.29234063246555814


In [37]:
import pandas as pd


def safe_text(x):
    if x is None:
        return ""

    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass

    return str(x)


def recursive_char_length(x):
    """
    Count characters inside nested strings/lists/tuples/dicts.
    """

    if x is None:
        return 0

    # Strings
    if isinstance(x, str):
        return len(x)

    # Dictionaries
    if isinstance(x, dict):
        total = 0

        for key, value in x.items():
            total += recursive_char_length(key)
            total += recursive_char_length(value)

        return total

    # Lists / tuples / sets
    if isinstance(x, (list, tuple, set)):
        return sum(
            recursive_char_length(item)
            for item in x
        )

    # Fallback
    return len(str(x))


def recursive_item_count(x):
    """
    Count top-level items separately from character length.
    """

    if x is None:
        return 0

    if isinstance(x, (list, tuple, set, dict)):
        return len(x)

    if isinstance(x, str):
        return int(bool(x.strip()))

    return 1

In [38]:
history_cols = [
    "previous_messages",
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
]


for col in history_cols:

    structured_df[f"{col}_count"] = (
        taxonomy_df[col]
        .map(recursive_item_count)
    )

    structured_df[f"{col}_chars"] = (
        taxonomy_df[col]
        .map(recursive_char_length)
    )

In [39]:
check_cols = []

for col in history_cols:
    check_cols.extend([
        f"{col}_count",
        f"{col}_chars",
    ])


display(
    structured_df[
        check_cols
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
previous_messages_count,1776.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
previous_messages_chars,1776.0,1.724099,0.567082,1.0,1.0,2.0,2.0,3.0
previous_tool_calls_count,1776.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
previous_tool_calls_chars,1776.0,1.334459,0.471934,1.0,1.0,1.0,2.0,2.0
previous_tool_results_count,1776.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
previous_tool_results_chars,1776.0,1.334459,0.471934,1.0,1.0,1.0,2.0,2.0
previous_user_messages_count,1776.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
previous_user_messages_chars,1776.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
previous_assistant_messages_count,1776.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
previous_assistant_messages_chars,1776.0,1.135135,0.341964,1.0,1.0,1.0,1.0,2.0


In [40]:
for col in history_cols:

    print("\n", "=" * 80)
    print(col)
    print("=" * 80)

    for idx in range(3):

        raw = taxonomy_df.loc[idx, col]

        print("RAW:")
        print(repr(raw)[:500])

        print(
            "COUNT:",
            structured_df.loc[
                idx,
                f"{col}_count"
            ]
        )

        print(
            "CHARS:",
            structured_df.loc[
                idx,
                f"{col}_chars"
            ]
        )

        print("-" * 40)


previous_messages
RAW:
6
COUNT: 1
CHARS: 1
----------------------------------------
RAW:
4
COUNT: 1
CHARS: 1
----------------------------------------
RAW:
8
COUNT: 1
CHARS: 1
----------------------------------------

previous_tool_calls
RAW:
3
COUNT: 1
CHARS: 1
----------------------------------------
RAW:
2
COUNT: 1
CHARS: 1
----------------------------------------
RAW:
4
COUNT: 1
CHARS: 1
----------------------------------------

previous_tool_results
RAW:
3
COUNT: 1
CHARS: 1
----------------------------------------
RAW:
2
COUNT: 1
CHARS: 1
----------------------------------------
RAW:
4
COUNT: 1
CHARS: 1
----------------------------------------

previous_user_messages
RAW:
0
COUNT: 1
CHARS: 1
----------------------------------------
RAW:
0
COUNT: 1
CHARS: 1
----------------------------------------
RAW:
0
COUNT: 1
CHARS: 1
----------------------------------------

previous_assistant_messages
RAW:
0
COUNT: 1
CHARS: 1
----------------------------------------
RAW:
0
COUNT: 1
CHARS: 1
-

In [41]:
# ============================================================
# Correct structural history features
# ============================================================

count_cols = [
    "previous_messages",
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
]

for col in count_cols:
    structured_df[col] = (
        pd.to_numeric(
            taxonomy_df[col],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )

In [42]:
# ============================================================
# Correct structural history features
# ============================================================

count_cols = [
    "previous_messages",
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
]

for col in count_cols:
    structured_df[col] = (
        pd.to_numeric(
            taxonomy_df[col],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )

In [43]:
bad_cols = []

for col in count_cols:
    bad_cols.extend([
        f"{col}_chars",
        f"{col}_count",
    ])

structured_df = structured_df.drop(
    columns=[
        c for c in bad_cols
        if c in structured_df.columns
    ],
    errors="ignore",
)

In [44]:
structured_df["current_char_length"] = (
    taxonomy_df["current_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

structured_df["current_word_count"] = (
    taxonomy_df["current_text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

structured_df["context_char_length"] = (
    taxonomy_df["context_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

structured_df["context_word_count"] = (
    taxonomy_df["context_text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

In [45]:
feature_columns = [
    # categorical
    "current_role",
    "current_tool",
    "previous_tool",

    # trajectory position / type
    "message_index",
    "is_tool_call",

    # current/context size
    "current_char_length",
    "current_word_count",
    "context_char_length",
    "context_word_count",

    # ACTUAL historical counts
    "previous_messages",
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",

    # relational
    "num_previous_tool_calls",
    "current_tool_previous_count",
    "same_tool_as_previous",
    "current_action_seen_before",

    # state/error indicators
    "previous_results_have_error",
    "context_has_error_signal",
    "has_previous_tool_result",
    "has_previous_tool_call",
    "has_previous_user_message",
]

In [46]:
print(
    structured_df[
        [
            "previous_tool_calls",
            "num_previous_tool_calls",
        ]
    ].corr()
)

print(
    (
        structured_df["previous_tool_calls"]
        == structured_df["num_previous_tool_calls"]
    ).mean()
)

                         previous_tool_calls  num_previous_tool_calls
previous_tool_calls                 1.000000                 0.153453
num_previous_tool_calls             0.153453                 1.000000
0.09177927927927929


In [48]:
structured_df["has_previous_tool_result"] = (
    structured_df["previous_tool_results"] > 0
).astype(int)

structured_df["has_previous_tool_call"] = (
    structured_df["previous_tool_calls"] > 0
).astype(int)

structured_df["has_previous_user_message"] = (
    structured_df["previous_user_messages"] > 0
).astype(int)

In [49]:
display(
    structured_df[
        [
            "previous_messages",
            "previous_tool_calls",
            "previous_tool_results",
            "previous_user_messages",
            "previous_assistant_messages",
            "current_char_length",
            "context_char_length",
            "same_tool_as_previous",
            "current_tool_previous_count",
        ]
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
previous_messages,1776.0,27.777027,34.570782,0.0,7.00,15.0,32.00,190.0
previous_tool_calls,1776.0,11.824887,15.742945,0.0,3.00,6.0,13.00,90.0
previous_tool_results,1776.0,11.824887,15.742945,0.0,3.00,6.0,13.00,90.0
previous_user_messages,1776.0,0.000000,0.000000,0.0,0.00,0.0,0.00,0.0
previous_assistant_messages,1776.0,4.309685,4.861326,0.0,1.00,3.0,6.00,28.0
current_char_length,1776.0,380.906532,475.129781,11.0,52.00,188.5,547.25,4358.0
context_char_length,1776.0,1036.108108,1810.166995,0.0,146.75,352.0,1000.50,18222.0
same_tool_as_previous,1776.0,0.110360,0.313427,0.0,0.00,0.0,0.00,1.0
current_tool_previous_count,1776.0,0.110360,0.313427,0.0,0.00,0.0,0.00,1.0


In [50]:
print(
    pd.crosstab(
        structured_df["same_tool_as_previous"],
        structured_df["current_tool_previous_count"],
    )
)

print(
    "Correlation:",
    structured_df[
        [
            "same_tool_as_previous",
            "current_tool_previous_count",
        ]
    ].corr().iloc[0, 1]
)

print(
    "Exact equality:",
    (
        structured_df["same_tool_as_previous"]
        == structured_df["current_tool_previous_count"]
    ).mean()
)

current_tool_previous_count     0    1
same_tool_as_previous                 
0                            1580    0
1                               0  196
Correlation: 1.0
Exact equality: 1.0


In [51]:
structured_df["current_tool_previous_count"] = 0

for group_id, idx in taxonomy_df.groupby(
    "group_id",
    sort=False,
).groups.items():

    counts = {}

    # Important: trajectory order
    ordered_idx = taxonomy_df.loc[
        idx
    ].sort_values(
        "message_index"
    ).index

    for i in ordered_idx:

        tool = structured_df.loc[
            i,
            "current_tool"
        ]

        if tool == "NO_TOOL":
            structured_df.loc[
                i,
                "current_tool_previous_count"
            ] = 0
            continue

        previous_count = counts.get(tool, 0)

        structured_df.loc[
            i,
            "current_tool_previous_count"
        ] = previous_count

        counts[tool] = previous_count + 1

In [52]:
structured_df["current_action_seen_before"] = (
    structured_df["current_tool_previous_count"] > 0
).astype(int)

In [53]:
display(
    structured_df[
        [
            "current_tool",
            "previous_tool",
            "same_tool_as_previous",
            "current_tool_previous_count",
            "current_action_seen_before",
        ]
    ].head(50)
)

,current_tool,previous_tool,same_tool_as_previous,current_tool_previous_count,current_action_seen_before
0,NO_TOOL,search,0,0,0
1,NO_TOOL,search,0,0,0
2,NO_TOOL,NO_TOOL,0,0,0
3,NO_TOOL,NO_TOOL,0,0,0
4,NO_TOOL,NO_TOOL,0,0,0
5,NO_TOOL,NO_TOOL,0,0,0
6,NO_TOOL,search,0,0,0
7,NO_TOOL,NO_TOOL,0,0,0
8,search,search,1,0,0
9,search,search,1,1,1


In [54]:
display(
    structured_df[
        [
            "same_tool_as_previous",
            "current_tool_previous_count",
            "current_action_seen_before",
        ]
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
same_tool_as_previous,1776.0,0.11036,0.313427,0.0,0.0,0.0,0.0,1.0
current_tool_previous_count,1776.0,2.28491,7.844468,0.0,0.0,0.0,0.0,64.0
current_action_seen_before,1776.0,0.20214,0.401709,0.0,0.0,0.0,0.0,1.0


In [55]:
structured_df = structured_df.drop(
    columns=["previous_user_messages"],
)

In [57]:
structured_df = structured_df.drop(
    columns=["has_previous_user_message"],
    errors="ignore",
)

In [58]:
constant_cols = [
    col
    for col in structured_df.columns
    if structured_df[col].nunique(dropna=False) <= 1
]

print("Constant features:")
print(constant_cols)

Constant features:
['previous_results_have_error']


In [59]:
structured_df = structured_df.drop(
    columns=constant_cols
)

In [60]:
cols = [
    "same_tool_as_previous",
    "current_tool_previous_count",
    "current_action_seen_before",
]

print(structured_df[cols].corr())

print(
    "Exact equality:",
    (
        structured_df["same_tool_as_previous"]
        == structured_df["current_tool_previous_count"]
    ).mean()
)

                             same_tool_as_previous  \
same_tool_as_previous                     1.000000   
current_tool_previous_count               0.546308   
current_action_seen_before                0.610249   

                             current_tool_previous_count  \
same_tool_as_previous                           0.546308   
current_tool_previous_count                     1.000000   
current_action_seen_before                      0.578849   

                             current_action_seen_before  
same_tool_as_previous                          0.610249  
current_tool_previous_count                    0.578849  
current_action_seen_before                     1.000000  
Exact equality: 0.8012387387387387


In [64]:
print(
    structured_df[
        [
            "previous_tool_calls",
            "num_previous_tool_calls",
        ]
    ].describe()
)

display(
    structured_df[
        [
            "message_index",
            "current_tool",
            "previous_tool_calls",
            "num_previous_tool_calls",
        ]
    ].head(50)
)

       previous_tool_calls  num_previous_tool_calls
count          1776.000000              1776.000000
mean             11.824887                 0.612613
std              15.742945                 0.487291
min               0.000000                 0.000000
25%               3.000000                 0.000000
50%               6.000000                 1.000000
75%              13.000000                 1.000000
max              90.000000                 1.000000


,message_index,current_tool,previous_tool_calls,num_previous_tool_calls
0,8,NO_TOOL,3,1
1,6,NO_TOOL,2,1
2,10,NO_TOOL,4,0
3,2,NO_TOOL,0,0
4,2,NO_TOOL,0,0
5,2,NO_TOOL,0,0
6,6,NO_TOOL,2,1
7,4,NO_TOOL,1,0
8,8,search,3,1
9,10,search,4,1


In [65]:
print(
    "Exact equality:",
    (
        structured_df["previous_tool_calls"]
        == structured_df["previous_tool_results"]
    ).mean()
)

print(
    pd.crosstab(
        structured_df["previous_tool_calls"],
        structured_df["previous_tool_results"],
    )
)

Exact equality: 1.0
previous_tool_results   0    1    2    3    4   5   6   7   8   9   ...  81  \
previous_tool_calls                                                 ...       
0                      101    0    0    0    0   0   0   0   0   0  ...   0   
1                        0  114    0    0    0   0   0   0   0   0  ...   0   
2                        0    0  146    0    0   0   0   0   0   0  ...   0   
3                        0    0    0  196    0   0   0   0   0   0  ...   0   
4                        0    0    0    0  175   0   0   0   0   0  ...   0   
...                    ...  ...  ...  ...  ...  ..  ..  ..  ..  ..  ...  ..   
86                       0    0    0    0    0   0   0   0   0   0  ...   0   
87                       0    0    0    0    0   0   0   0   0   0  ...   0   
88                       0    0    0    0    0   0   0   0   0   0  ...   0   
89                       0    0    0    0    0   0   0   0   0   0  ...   0   
90                       0    0 

In [67]:
feature_cols = [
    col for col in feature_columns 
    if col not in constant_cols
]

In [68]:
feature_cols

['current_role',
 'current_tool',
 'previous_tool',
 'message_index',
 'is_tool_call',
 'current_char_length',
 'current_word_count',
 'context_char_length',
 'context_word_count',
 'previous_messages',
 'previous_tool_calls',
 'previous_tool_results',
 'previous_user_messages',
 'previous_assistant_messages',
 'num_previous_tool_calls',
 'current_tool_previous_count',
 'same_tool_as_previous',
 'current_action_seen_before',
 'context_has_error_signal',
 'has_previous_tool_result',
 'has_previous_tool_call',
 'has_previous_user_message']

In [69]:
structured_df = structured_df.rename(
    columns={
        "num_previous_tool_calls":
            "parsed_tool_calls_in_context"
    }
)

In [70]:
drop_cols = [
    "previous_tool_results",
    "previous_user_messages",
    "has_previous_user_message",
]

structured_df = structured_df.drop(
    columns=[
        c for c in drop_cols
        if c in structured_df.columns
    ],
)

In [73]:
constant_cols = [
    col
    for col in structured_df.columns
    if structured_df[col].nunique(
        dropna=False
    ) <= 1
]

print("Constant columns:")
print(constant_cols)

structured_df = structured_df.drop(
    columns=constant_cols
)

Constant columns:
[]


In [74]:
feature_columns = [
    # ---------------------------------
    # Current action structure
    # ---------------------------------
    "current_role",
    "current_tool",

    "message_index",
    "is_tool_call",

    "current_char_length",
    "current_word_count",

    # ---------------------------------
    # Available trajectory context
    # ---------------------------------
    "context_char_length",
    "context_word_count",

    "previous_messages",
    "previous_tool_calls",
    "previous_assistant_messages",

    "parsed_tool_calls_in_context",

    # ---------------------------------
    # Relational features
    # ---------------------------------
    "previous_tool",
    "same_tool_as_previous",
    "current_tool_previous_count",
    "current_action_seen_before",

    # ---------------------------------
    # Error / state features
    # ---------------------------------
    "previous_results_have_error",
    "context_has_error_signal",

    "has_previous_tool_result",
    "has_previous_tool_call",
]

In [77]:
import re


ERROR_PATTERNS = [
    r"\berror\b",
    r"\bfailed\b",
    r"\bfailure\b",
    r"\binvalid\b",
    r"\bexception\b",
    r"\bnot found\b",
    r"\bunauthorized\b",
    r"\bforbidden\b",
    r"\bdenied\b",
]


def has_error_signal(text):
    if not isinstance(text, str):
        return 0

    text = text.lower()

    return int(
        any(
            re.search(pattern, text)
            for pattern in ERROR_PATTERNS
        )
    )


structured_df["previous_results_have_error"] = (
    taxonomy_df["context_text"]
    .fillna("")
    .apply(has_error_signal)
)

In [78]:
structured_df["context_has_error_signal"] = (
    taxonomy_df["context_text"]
    .fillna("")
    .apply(has_error_signal)
)

In [79]:
for col in [
    "previous_results_have_error",
    "context_has_error_signal",
]:
    print(
        col,
        col in structured_df.columns
    )

previous_results_have_error True
context_has_error_signal True


In [80]:
missing_features = [
    c
    for c in feature_columns
    if c not in structured_df.columns
]

print("Missing:", missing_features)

assert len(missing_features) == 0

Missing: []


In [81]:
print("Shape:", structured_df[feature_columns].shape)

print("\nDtypes:")
print(
    structured_df[
        feature_columns
    ].dtypes
)

print("\nUnique values:")
print(
    structured_df[
        feature_columns
    ].nunique()
)

print("\nMissing values:")
print(
    structured_df[
        feature_columns
    ].isna().sum()
)

Shape: (1776, 20)

Dtypes:
current_role                    object
current_tool                    object
message_index                    int64
is_tool_call                     int64
current_char_length              int64
current_word_count               int64
context_char_length              int64
context_word_count               int64
previous_messages                int64
previous_tool_calls              int64
previous_assistant_messages      int64
parsed_tool_calls_in_context     int64
previous_tool                   object
same_tool_as_previous            int64
current_tool_previous_count      int64
current_action_seen_before       int64
previous_results_have_error      int64
context_has_error_signal         int64
has_previous_tool_result         int64
has_previous_tool_call           int64
dtype: object

Unique values:
current_role                      2
current_tool                     87
message_index                   138
is_tool_call                      2
current_char_length

In [82]:
train_group_set = set(groups_train)
test_group_set = set(groups_test)

train_mask = taxonomy_df["group_id"].isin(
    train_group_set
)

test_mask = taxonomy_df["group_id"].isin(
    test_group_set
)

structured_train = (
    structured_df.loc[
        train_mask,
        feature_columns,
    ]
    .reset_index(drop=True)
)

structured_test = (
    structured_df.loc[
        test_mask,
        feature_columns,
    ]
    .reset_index(drop=True)
)

y_train_structured = (
    taxonomy_df.loc[
        train_mask,
        "family_label",
    ]
    .reset_index(drop=True)
)

y_test_structured = (
    taxonomy_df.loc[
        test_mask,
        "family_label",
    ]
    .reset_index(drop=True)
)

print(
    "Train:",
    structured_train.shape,
)

print(
    "Test:",
    structured_test.shape,
)

assert len(structured_train) == 1489
assert len(structured_test) == 287

assert np.array_equal(
    y_train_structured.to_numpy(),
    np.asarray(y_train),
)

assert np.array_equal(
    y_test_structured.to_numpy(),
    np.asarray(y_test),
)

print("✓ Exact original split recovered")

Train: (1489, 20)
Test: (287, 20)
✓ Exact original split recovered


In [83]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

categorical_features = [
    "current_role",
    "current_tool",
    "previous_tool",
]

numeric_features = [
    col
    for col in feature_columns
    if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    ),
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    ),
                ),
            ]),
            categorical_features,
        ),
        (
            "numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
            ]),
            numeric_features,
        ),
    ]
)

In [84]:
s1_model = Pipeline([
    (
        "preprocessor",
        preprocessor,
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=5000,
            class_weight=None,
            random_state=42,
        ),
    ),
])

s1_model.fit(
    structured_train,
    y_train_structured,
)

s1_pred = s1_model.predict(
    structured_test
)

In [85]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

family_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

print(
    classification_report(
        y_test_structured,
        s1_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

s1_accuracy = accuracy_score(
    y_test_structured,
    s1_pred,
)

s1_balanced_accuracy = (
    balanced_accuracy_score(
        y_test_structured,
        s1_pred,
    )
)

s1_macro_f1 = f1_score(
    y_test_structured,
    s1_pred,
    average="macro",
    zero_division=0,
)

s1_weighted_f1 = f1_score(
    y_test_structured,
    s1_pred,
    average="weighted",
    zero_division=0,
)

print("Accuracy:", s1_accuracy)
print(
    "Balanced accuracy:",
    s1_balanced_accuracy,
)
print(
    "Macro F1:",
    s1_macro_f1,
)
print(
    "Weighted F1:",
    s1_weighted_f1,
)

print("\nConfusion matrix:")

print(
    confusion_matrix(
        y_test_structured,
        s1_pred,
        labels=[0, 1, 2, 3, 4],
    )
)

                       precision    recall  f1-score   support

       workflow_error     0.5854    0.5217    0.5517       138
     constraint_error     0.5238    0.4714    0.4962        70
       tool_use_error     0.3611    0.3421    0.3514        38
grounding_state_error     0.1579    0.3000    0.2069        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.4634       287
            macro avg     0.4756    0.4361    0.4476       287
         weighted avg     0.5023    0.4634    0.4787       287

Accuracy: 0.4634146341463415
Balanced accuracy: 0.43614550209515884
Macro F1: 0.44755831797574075
Weighted F1: 0.47867731520513596

Confusion matrix:
[[72 17 19 29  1]
 [27 33  3  7  0]
 [ 9  7 13  9  0]
 [13  6  1  9  1]
 [ 2  0  0  3  6]]


In [86]:
X_train_struct = (
    s1_model
    .named_steps["preprocessor"]
    .transform(structured_train)
)

X_test_struct = (
    s1_model
    .named_steps["preprocessor"]
    .transform(structured_test)
)

print(X_train_struct.shape)
print(X_test_struct.shape)

(1489, 202)
(287, 202)


In [94]:
from scripts.taxonomy_dataset import transformer_train_df, transformer_test_df, reason_model

# ============================================================
# S2: CURRENT-ONLY SEMANTIC EMBEDDINGS
# ============================================================

train_current_texts = (
    transformer_train_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

test_current_texts = (
    transformer_test_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

X_train_current_embeddings = reason_model.encode(
    train_current_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

X_test_current_embeddings = reason_model.encode(
    test_current_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Train embeddings:", X_train_current_embeddings.shape)
print("Test embeddings:", X_test_current_embeddings.shape)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Train embeddings: (1489, 384)
Test embeddings: (287, 384)


In [95]:
assert len(X_train_current_embeddings) == len(structured_train)
assert len(X_test_current_embeddings) == len(structured_test)

assert len(X_train_current_embeddings) == len(y_train)
assert len(X_test_current_embeddings) == len(y_test)

print("✓ Everything aligned")

✓ Everything aligned


In [97]:
structured_features = categorical_features + numeric_features

missing = [
    col for col in structured_features
    if col not in structured_df.columns
]

print("Missing:", missing)
assert not missing

Missing: []


In [99]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

structured_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
    ],
    remainder="drop",
)

In [100]:
X_train_structured = structured_preprocessor.fit_transform(
    structured_train
)

X_test_structured = structured_preprocessor.transform(
    structured_test
)

print("Train:", X_train_structured.shape)
print("Test:", X_test_structured.shape)

Train: (1489, 202)
Test: (287, 202)


In [102]:
from scipy.sparse import csr_matrix, hstack

X_train_fused = hstack([
    csr_matrix(X_train_current_embeddings),
    X_train_structured,
]).tocsr()

X_test_fused = hstack([
    csr_matrix(X_test_current_embeddings),
    X_test_structured,
]).tocsr()

print("Current embedding:", X_train_current_embeddings.shape)
print("Structured:", X_train_structured.shape)
print("Fused:", X_train_fused.shape)

Current embedding: (1489, 384)
Structured: (1489, 202)
Fused: (1489, 586)


In [103]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

fusion_classifier = LogisticRegression(
    max_iter=5000,
    class_weight=None,
    random_state=42,
)

fusion_classifier.fit(
    X_train_fused,
    y_train,
)

y_pred_fused = fusion_classifier.predict(X_test_fused)

print("=" * 80)
print("SEMANTIC CURRENT + STRUCTURED TRAJECTORY")
print("=" * 80)

print(
    classification_report(
        y_test,
        y_pred_fused,
        target_names=family_names[:5],
        digits=4,
    )
)

accuracy = accuracy_score(y_test, y_pred_fused)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred_fused)
macro_f1 = f1_score(y_test, y_pred_fused, average="macro")
weighted_f1 = f1_score(y_test, y_pred_fused, average="weighted")

print("Accuracy:", accuracy)
print("Balanced accuracy:", balanced_accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_fused))

SEMANTIC CURRENT + STRUCTURED TRAJECTORY
                       precision    recall  f1-score   support

       workflow_error     0.5840    0.5290    0.5551       138
     constraint_error     0.5152    0.4857    0.5000        70
       tool_use_error     0.3243    0.3158    0.3200        38
grounding_state_error     0.3137    0.5333    0.3951        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.4913       287
            macro avg     0.4974    0.4819    0.4804       287
         weighted avg     0.5109    0.4913    0.4968       287

Accuracy: 0.4912891986062718
Balanced accuracy: 0.4818554290865503
Macro F1: 0.4803547511222783
Weighted F1: 0.4967511682645153

Confusion matrix:
[[73 22 21 21  1]
 [28 34  3  5  0]
 [14  5 12  7  0]
 [ 7  5  1 16  1]
 [ 3  0  0  2  6]]


In [105]:
baseline_macro_f1 = 0.490268
baseline_accuracy = 0.515679

print("Δ Macro F1:", macro_f1 - baseline_macro_f1)
print("Δ Accuracy:", accuracy - baseline_accuracy)

Δ Macro F1: -0.009913248877721659
Δ Accuracy: -0.024389801393728205


Yes. For `8_structured_relational_features`, I would finish the notebook with a concise **Results / Findings** section that records both the structured-only experiment and the fusion experiment.

You can paste this directly into a Markdown cell:

```markdown
# 8. Structured / Relational Trajectory Features — Results

## Objective

This experiment tested whether explicit structural information about an agent trajectory can improve failure-family classification beyond semantic information contained in the current message.

The hypothesis was that some failure families are relational rather than purely semantic. For example, an error may depend on whether:

- a tool was previously called,
- the same action has already been attempted,
- a previous tool result contained an error,
- the current action repeats an earlier action,
- the current message occurs late in a trajectory,
- or the current response depends on previous trajectory state.

The experiment therefore constructed structured trajectory features and evaluated them independently and jointly with current-message semantic embeddings.

---

## Dataset

The same group-safe train/test split used in previous experiments was retained:

- Total examples: **1,776**
- Training examples: **1,489**
- Test examples: **287**
- Group overlap between train and test: **0**

Failure-family distribution:

| Failure family | Count |
|---|---:|
| workflow_error | 798 |
| constraint_error | 387 |
| tool_use_error | 275 |
| grounding_state_error | 274 |
| reasoning_value_error | 42 |

The dataset remains strongly imbalanced, particularly for `reasoning_value_error`.

---

## Structured Representation

The structured representation contained information about the current trajectory state rather than the semantic content of the message.

Features included:

- current role
- current tool
- previous tool
- message position
- whether the current message is a tool call
- current/context length
- number of previous messages
- number of previous tool calls
- number of previous assistant messages
- presence of previous tool calls/results
- repeated-tool indicators
- number of previous uses of the current tool
- whether the current action had been seen before
- previous-result error signals
- context error signals

Categorical features were one-hot encoded and numerical features were standardized.

After preprocessing:

- Semantic current-message embedding: **384 dimensions**
- Structured trajectory representation: **202 dimensions**
- Fused representation: **586 dimensions**

---

## Important Data Validation Finding

Initial inspection revealed that several `previous_*` columns had been incorrectly interpreted as sequence-like values when they were already aggregate counts.

For example, values such as:

    previous_tool_calls = 3

were initially treated as strings/containers, producing meaningless derived features.

After correcting this interpretation, the trajectory features showed realistic distributions.

Additional validation showed that:

- `previous_tool_calls` and `previous_tool_results` were exactly equal in this dataset.
- `same_tool_as_previous` and `current_tool_previous_count` were related but not identical.
- `current_action_seen_before` correctly represented whether the current tool/action had occurred earlier in the trajectory.

This validation step was necessary before evaluating the structured representation.

---

# Experiment 1 — Structured Features Only

A Logistic Regression classifier was trained using only the structured / relational trajectory features.

### Results

| Metric | Score |
|---|---:|
| Accuracy | **0.4634** |
| Balanced Accuracy | **0.4361** |
| Macro F1 | **0.4476** |
| Weighted F1 | **0.4787** |

Per-class results:

| Failure family | Precision | Recall | F1 |
|---|---:|---:|---:|
| workflow_error | 0.5854 | 0.5217 | 0.5517 |
| constraint_error | 0.5238 | 0.4714 | 0.4962 |
| tool_use_error | 0.3611 | 0.3421 | 0.3514 |
| grounding_state_error | 0.1579 | 0.3000 | 0.2069 |
| reasoning_value_error | 0.7500 | 0.5455 | 0.6316 |

### Interpretation

Structured trajectory information alone contains substantial predictive signal.

A Macro F1 of **0.4476** shows that failure families are not determined exclusively by message semantics. Information about trajectory position, tool usage, repetition, and previous state can help identify the type of failure.

However, structured features alone remained weaker than the previously established current-message semantic embedding baseline.

---

# Experiment 2 — Semantic + Structured Feature Fusion

The next experiment tested whether trajectory information provided complementary information beyond current-message semantics.

The representations were concatenated:

    current semantic embedding: 384 dimensions
                         +
    structured trajectory:      202 dimensions
                         =
    fused representation:       586 dimensions

The same Logistic Regression classifier was then trained on the fused representation.

### Results

| Metric | Score |
|---|---:|
| Accuracy | **0.4913** |
| Balanced Accuracy | **0.4819** |
| Macro F1 | **0.4804** |
| Weighted F1 | **0.4968** |

Per-class results:

| Failure family | Precision | Recall | F1 |
|---|---:|---:|---:|
| workflow_error | 0.5840 | 0.5290 | 0.5551 |
| constraint_error | 0.5152 | 0.4857 | 0.5000 |
| tool_use_error | 0.3243 | 0.3158 | 0.3200 |
| grounding_state_error | 0.3137 | 0.5333 | 0.3951 |
| reasoning_value_error | 0.7500 | 0.5455 | 0.6316 |

---

# Representation Comparison

| Representation | Accuracy | Balanced Accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| Current semantic embedding | **0.5157** | 0.4814 | **0.4903** | **0.5215** |
| Structured trajectory only | 0.4634 | 0.4361 | 0.4476 | 0.4787 |
| Current + structured | 0.4913 | **0.4819** | 0.4804 | 0.4968 |

Relative to the current-message semantic baseline, naive feature fusion produced approximately:

    Δ Accuracy          = -0.0244
    Δ Balanced Accuracy = +0.0005
    Δ Macro F1          = -0.0099
    Δ Weighted F1       = -0.0247

Therefore, concatenating structured trajectory features with semantic embeddings did **not improve aggregate classification performance**.

---

# Key Finding

The experiment supports two findings simultaneously.

### 1. Trajectory structure contains predictive information

Structured features alone achieved:

    Macro F1 = 0.4476

Therefore, trajectory structure is meaningfully associated with failure family.

Failure classification is not purely a text-classification problem.

### 2. Naive early fusion does not provide complementary aggregate improvement

The strongest simple representation remained:

    Current-message semantic embedding
    Macro F1 = 0.4903

while:

    Current + structured
    Macro F1 = 0.4804

Thus, simply concatenating semantic and structured representations does not allow a linear classifier to exploit their relationship effectively.

---

# Class-Level Finding

The aggregate result hides an important class-level effect.

Structured information appears particularly useful for some trajectory-dependent classes.

For example, the fused model achieved:

    grounding_state_error
    Recall = 0.5333
    F1     = 0.3951

This is stronger than the structured-only result for that class:

    F1 = 0.2069

The fused model also achieved:

    tool_use_error
    F1 = 0.3200

This suggests that different failure families depend on different information sources.

Some failures are primarily recognizable from the current message, while others benefit from information about the surrounding trajectory.

---

# Research Conclusion

The results suggest that failure-family classification contains both:

1. **semantic signal** — what the current message says or does, and
2. **relational/trajectory signal** — how the current action relates to previous actions and system state.

However, these signals cannot necessarily be exploited optimally through simple feature concatenation.

A linear model over:

    [semantic embedding ; structured features]

assumes relatively simple additive relationships between the two representations.

Many agent failures are likely conditional interactions, such as:

    "This tool call is problematic because
     the same action has already failed earlier."

or:

    "This answer is unsupported because
     the previous tool result does not establish the claim."

Such relationships require interaction between semantic content and trajectory state.

Therefore, the negative fusion result should not be interpreted as evidence that trajectory information is useless.

Instead, the experiment suggests:

> Structured trajectory information is independently predictive, but naive linear early fusion does not improve over the strongest semantic baseline. Future models should explicitly learn interactions between semantic and trajectory representations.

---

# Final Result of Experiment 8

**Best model remains:**

    Current-message semantic embedding
    + Logistic Regression

    Accuracy  = 0.5157
    Macro F1  = 0.4903

**Structured-only:**

    Accuracy  = 0.4634
    Macro F1  = 0.4476

**Semantic + structured fusion:**

    Accuracy  = 0.4913
    Macro F1  = 0.4804

The structured-feature hypothesis is therefore **partially supported**:

- trajectory features contain useful predictive information;
- some failure classes benefit from relational information;
- but simple early feature fusion does not outperform the semantic baseline.

## Next Experiment

The next experiment should test whether the two information sources can be combined more effectively using **late fusion / stacking** or a **nonlinear learned fusion model**, rather than direct feature concatenation.
```

This closes notebook 8 cleanly: it records not only that fusion failed to beat `0.4903`, but **why that negative result is scientifically useful** and gives you a justified transition to the next experiment.
